<a href="https://colab.research.google.com/github/anushabershilla/Data-science-ML/blob/main/Document_Question_Answering_System_using_RAG_and_ChromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
#import libraries
!pip install chromadb sentence-transformers -q

#import libraries
import chromadb
from sentence_transformers import SentenceTransformer

#Load Embedding model

model=SentenceTransformer("all-miniLM-L6-v2")

#Create chunks

document=["""The leave policy is created to give employees a clear picture of the number of leaves an
employee is allowed to take annually. Furthermore, it should also mention public holidays.
Furthermore, the leave policy must also mention the percentage of salary cut that an
employee will see in the case of unpaid leaves or half-days. All regular employees are entitled
for 18 Earned Leaves in a calendar year. However, if the employee has joined the
organization in the middle of the calendar year, then the quantum of Earned Leaves will be
pro rata basis.
In case of new joiners/employee transfer, following rules of leave credit will be applicable:
a) Employee joining/transferred before or on 10th of the month: 1.5 Leaves
b) Employee joining/transferred between 11th to 18th of the month: 1 Leave
c) Employee joining/transferred after 20th of the month: No Credit
d) We give 0.5 leave to the employee on his birthday.
e) For Initial Three Months No Paid Leave will be allowed.
f) In a Notice Period No Leave allow.
g) All leaves shall be calculated from date of joining of the employee.
h) The maximum period of casual leave which a staff is allowed to avail is 12 days
in calendar year subject to a maximum of 3 days at a time, normally. The limit of
3 days at a time may be relaxed in special circumstances at the discretion of the
HR Department.
i) In case any staff remains absent from duty for more than 5 days without any
intimation to the concerned authorities his/her contract is liable to be
terminated by the concerned authority.
All Leaves will be credited to the concerned at the beginning of each year, and the leaves
remaining at the end of the year will automatically lapse. We have carrying forward leaves
to next year limits of 6 Leaves and Encashment to other leave balance for employees.
Legally, India has three national holidays where no organization is allowed to be open without
permission. However, organizations like factories, hospitals, travel agencies, etc. are exempt
from this and are allowed to work for 24 hours a day. However, under the Factories Act, 1948,
they must be paid for those days and should be paid for overtime.
S. No. Holiday Date"""
]

chunks=[chunk.strip() for chunk in document[0].split("\n") if chunk.strip()]
print(chunks)

#Generate Embadding

embaddings=model.encode(chunks)

#Create chroma client

client=chromadb.Client()

#Create collection

collection=client.get_or_create_collection(name="hr_policy")

#insert Data
# Generate unique IDs for each chunk
ids = [f"doc_{i}" for i in range(len(chunks))]
collection.add(documents=chunks,
               embeddings=embaddings.tolist(),
               ids=ids)

#User Query

query="Howmany casual leaves available for employee in year"

#Query Embadding

query_embadding =model.encode([query])

#Serach chroma or Retrive chunks with high cosine score

results=collection.query(query_embadding.tolist(),n_results=3)

#View Result
print(results)
retrieved_docs = results["documents"][0]

print(retrieved_docs)

#create Context

context="\n".join(retrieved_docs)
print(context)

#Create prompt

prompt=f"""
Answer the question using the context below.
Context:
{context}
Question:
{query}
Answer:
"""
print(prompt)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

['The leave policy is created to give employees a clear picture of the number of leaves an', 'employee is allowed to take annually. Furthermore, it should also mention public holidays.', 'Furthermore, the leave policy must also mention the percentage of salary cut that an', 'employee will see in the case of unpaid leaves or half-days. All regular employees are entitled', 'for 18 Earned Leaves in a calendar year. However, if the employee has joined the', 'organization in the middle of the calendar year, then the quantum of Earned Leaves will be', 'pro rata basis.', 'In case of new joiners/employee transfer, following rules of leave credit will be applicable:', 'a) Employee joining/transferred before or on 10th of the month: 1.5 Leaves', 'b) Employee joining/transferred between 11th to 18th of the month: 1 Leave', 'c) Employee joining/transferred after 20th of the month: No Credit', 'd) We give 0.5 leave to the employee on his birthday.', 'e) For Initial Three Months No Paid Leave will b